In [1]:
"""
Motor Fault Detection - Best Model: Random Forest
✅ class_weight="balanced" to fix class imbalance bias
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ─── 1. LOAD DATA ─────────────────────────────────────────────────────────────
df = pd.read_csv("motor_dataset.csv")

print("=" * 55)
print("  Motor Fault Detection — Random Forest Classifier")
print("=" * 55)
print(f"\nDataset shape : {df.shape}")
print(f"\nClass distribution:")
print(df["fault"].value_counts().to_string())

# ─── 2. PREPROCESS ────────────────────────────────────────────────────────────
features = ["rpm", "current", "temperature", "vibration"]
X = df[features]
y = df["fault"]

le     = LabelEncoder()
y_enc  = le.fit_transform(y)

scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc          # keeps class ratio in both splits
)

print(f"\nTrain: {len(X_train)} samples | Test: {len(X_test)} samples")

# ─── 3. TRAIN — with class_weight="balanced" to fix bias ──────────────────────
# ⚠️  Without this, model predicts NORMAL (81%) almost always.
# ✅  "balanced" auto-adjusts weights inversely to class frequency.
model = RandomForestClassifier(
    n_estimators=100,           # 100 trees in the forest
    class_weight="balanced",    # 🔥 KEY: fixes class imbalance bias
    random_state=42,
    n_jobs=-1                   # use all CPU cores
)

model.fit(X_train, y_train)
print("\n✅ Model trained successfully!")

# ─── 4. EVALUATE ──────────────────────────────────────────────────────────────
y_pred = model.predict(X_test)

print("\n" + "=" * 55)
print("  CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=le.classes_))

# 5-fold cross-validation for robustness
cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_scaled, y_enc, cv=cv, scoring="f1_macro")
print(f"Cross-Val Macro F1 : {scores.mean():.4f} ± {scores.std():.4f}")

# ─── 5. FEATURE IMPORTANCE ────────────────────────────────────────────────────
importances = model.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]
print("\nFeature Importance:")
for i in sorted_idx:
    bar = "█" * int(importances[i] * 40)
    print(f"  {features[i]:<14} {importances[i]:.4f}  {bar}")

# ─── 6. PREDICT A NEW SAMPLE ──────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  PREDICT ON NEW MOTOR READINGS")
print("=" * 55)

new_samples = pd.DataFrame({
    "rpm":         [1500, 950,  1200],
    "current":     [1.1,  2.5,  1.8],
    "temperature": [28,   55,   45],
    "vibration":   [0.15, 0.75, 0.50],
})
new_scaled = scaler.transform(new_samples)
predictions     = model.predict(new_scaled)
probabilities   = model.predict_proba(new_scaled)

for i, (pred, proba) in enumerate(zip(predictions, probabilities)):
    label = le.inverse_transform([pred])[0]
    conf  = proba.max() * 100
    print(f"  Sample {i+1}: {new_samples.iloc[i].to_dict()}")
    print(f"           → Predicted: {label}  (confidence: {conf:.1f}%)\n")

# ─── 7. VISUALIZATION ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
fig.suptitle("Motor Fault Detection — Random Forest (Best Model)", fontsize=15, fontweight="bold")
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

# Plot 1: Feature Importance
ax1 = fig.add_subplot(gs[0, 0])
feat_sorted = [features[i] for i in sorted_idx]
imp_sorted  = [importances[i] for i in sorted_idx]
bars = ax1.barh(feat_sorted[::-1], imp_sorted[::-1], color=COLORS[:len(features)])
ax1.set_xlabel("Importance Score")
ax1.set_title("🔍 Feature Importance", fontweight="bold")
for bar, val in zip(bars, imp_sorted[::-1]):
    ax1.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f"{val:.4f}", va="center", fontsize=9)

# Plot 2: Class Distribution (showing imbalance)
ax2 = fig.add_subplot(gs[0, 1])
dist = df["fault"].value_counts()
wedge_props = dict(edgecolor="white", linewidth=2)
ax2.pie(dist.values,
        labels=[f"{l}\n({v})" for l, v in zip(dist.index, dist.values)],
        autopct="%1.1f%%",
        colors=COLORS,
        wedgeprops=wedge_props,
        startangle=140)
ax2.set_title("⚠️  Class Imbalance\n(why balanced weight matters)", fontweight="bold")

# Plot 3: Confusion Matrix
ax3 = fig.add_subplot(gs[1, 0])
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax3, colorbar=False, cmap="Blues")
ax3.set_title("✅ Confusion Matrix (Test Set)", fontweight="bold")
ax3.tick_params(axis="x", rotation=25)

# Plot 4: Cross-Val Scores per fold
ax4 = fig.add_subplot(gs[1, 1])
fold_labels = [f"Fold {i+1}" for i in range(len(scores))]
bars = ax4.bar(fold_labels, scores, color=COLORS[:len(scores)], edgecolor="white", linewidth=1.5)
ax4.axhline(scores.mean(), color="red", linestyle="--", linewidth=1.5,
            label=f"Mean = {scores.mean():.4f}")
ax4.set_ylim(0, 1.05)
ax4.set_ylabel("Macro F1 Score")
ax4.set_title("📊 5-Fold Cross-Validation", fontweight="bold")
ax4.legend()
for bar, val in zip(bars, scores):
    ax4.text(bar.get_x() + bar.get_width()/2, val + 0.01,
             f"{val:.4f}", ha="center", fontsize=9, fontweight="bold")

plt.savefig("motor_random_forest_final.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Chart saved → motor_random_forest_final.png")
print("\n🎉 Done!")

  Motor Fault Detection — Random Forest Classifier

Dataset shape : (1000, 5)

Class distribution:
fault
NORMAL           816
OVERLOAD          63
BEARING_FAULT     58
OVERHEATING       35
STALL             28

Train: 800 samples | Test: 200 samples

✅ Model trained successfully!

  CLASSIFICATION REPORT
               precision    recall  f1-score   support

BEARING_FAULT       1.00      1.00      1.00        11
       NORMAL       1.00      1.00      1.00       163
  OVERHEATING       1.00      1.00      1.00         7
     OVERLOAD       1.00      1.00      1.00        13
        STALL       1.00      1.00      1.00         6

     accuracy                           1.00       200
    macro avg       1.00      1.00      1.00       200
 weighted avg       1.00      1.00      1.00       200

Cross-Val Macro F1 : 1.0000 ± 0.0000

Feature Importance:
  vibration      0.3054  ████████████
  rpm            0.2957  ███████████
  temperature    0.2029  ████████
  current        0.1960  ████

In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

# -------- LOAD DATA --------
df = pd.read_csv("motor_dataset.csv")

# -------- FEATURES & LABEL --------
X = df[["rpm", "current", "temperature", "vibration"]]
y = df["fault"]

# -------- SPLIT DATA --------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------- MODEL --------
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",   # VERY IMPORTANT for your case
    random_state=42
)

# -------- TRAIN --------
model.fit(X_train, y_train)

# -------- EVALUATE --------
y_pred = model.predict(X_test)

print("\n✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:\n")
print(classification_report(y_test, y_pred))

# -------- SAVE MODEL --------
joblib.dump(model, "model.pkl")

print("\n💾 Model saved as model.pkl")


✅ Accuracy: 1.0

📊 Classification Report:

               precision    recall  f1-score   support

BEARING_FAULT       1.00      1.00      1.00        10
       NORMAL       1.00      1.00      1.00       166
  OVERHEATING       1.00      1.00      1.00         7
     OVERLOAD       1.00      1.00      1.00         8
        STALL       1.00      1.00      1.00         9

     accuracy                           1.00       200
    macro avg       1.00      1.00      1.00       200
 weighted avg       1.00      1.00      1.00       200


💾 Model saved as model.pkl
